In [12]:
import joblib
import pandas as pd
from pathlib import Path
from typing import Dict, Any, List


MODEL_DIR = Path(r'D:\MINI_PROJECT_CLG\my final dataset') 

MODEL_FILE = MODEL_DIR / 'depression_predictor.joblib'
SCALER_FILE = MODEL_DIR / 'depression_scaler.joblib'


QUESTIONS_CONFIG: List[Dict[str, Any]] = [
    {"name": "no interest", "prompt": "Do you have little or no interest in doing things? (rate from 0-3)", "min": 0, "max": 3},
    {"name": "sleep", "prompt": "Do you have trouble falling asleep or sleeping too much? (rate from 0-3)", "min": 0, "max": 3},
    {"name": "appetite", "prompt": "Are you having a poor appetite or overeating problem? (rate from 0-3)", "min": 0, "max": 3},
    {"name": "on edge", "prompt": "Are you feeling nervous or on edge? (rate from 0-3)", "min": 0, "max": 3},
    {"name": "overthinking", "prompt": "Did you find yourself to be overthinking nowadays? (rate from 0-3)", "min": 0, "max": 3},
    {"name": "cannot cope", "prompt": "Did you find yourself unable to cope with all the things you had to do? (rate from 0-3)", "min": 0, "max": 3},
    {"name": "too much work", "prompt": "Are you feeling that too much work is piling up, making you feel overwhelmed? (rate from 0-3)", "min": 0, "max": 3},
    {"name": "stress", "prompt": "How much would you rate your stress level? (0-40)", "min": 0, "max": 40},
    {"name": "anxiety", "prompt": "How much would you rate your anxiety level? (0-27)", "min": 0, "max": 27},
]


def load_model_and_scaler(model_path: Path, scaler_path: Path) -> tuple:
    """Loads the joblib-saved model and scaler with error handling."""
    print("🤖 Loading model and scaler...")
    try:
        loaded_model = joblib.load(model_path)
        scaler = joblib.load(scaler_path)
        print("✅ Load successful.")
        return loaded_model, scaler
    except FileNotFoundError as e:
        print(f"❌ Error: Required file not found. Check path: {e.filename}")
        raise
    except Exception as e:
        print(f"❌ An error occurred during model loading: {e}")
        raise

def get_user_input(config: List[Dict[str, Any]]) -> pd.DataFrame:
    """Collects and validates user input, returning it as a DataFrame."""
    print("\n--- Please answer the following questions ---")
    data_row = {}
    
    for item in config:
        while True:
            try:
                raw_input = input(f"{item['prompt']}: ")
                value = int(raw_input.strip())
                
               
                if item['min'] <= value <= item['max']:
                    data_row[item['name']] = value
                    break
                else:
                    print(f"⚠️ Invalid input. Please enter a value between {item['min']} and {item['max']}.")
            except ValueError:
                print("⚠️ Invalid input. Please enter a whole number.")

    data = [list(data_row.values())]
    columns = list(data_row.keys())
    return pd.DataFrame(data, columns=columns)

def predict_depression(model, scaler, df: pd.DataFrame) -> bool:
    """Scales the input data and generates the prediction."""

    expected_cols = [c['name'] for c in QUESTIONS_CONFIG]
    if not list(df.columns) == expected_cols:
         raise ValueError("DataFrame columns do not match expected model input features.")

    print("\n--- Processing Data ---")
    
   
    df_scaled = pd.DataFrame(
        scaler.transform(df), 
        columns=df.columns
    )
    print("📈 Data scaled successfully.")
    
  
    prediction = model.predict(df_scaled)

    return prediction[0].item() == 1

def display_results(is_depressed: bool) -> None:
    """Prints the final, user-friendly prediction result."""
    print("\n" + "="*40)
    print("      *** DEPRESSION PREDICTION ***")
    print("="*40)
    
    if is_depressed:
        print("🚩 The person **shows signs of depression** (Prediction: 1).")
        print("    It is highly recommended to seek professional consultation.")
    else:
        print("✅ The person **does not show signs of depression** (Prediction: 0).")
        print("    Remember, this is not a substitute for professional medical advice.")
    
    print("="*40)


if __name__ == "__main__":
    try:
   
        model, scaler = load_model_and_scaler(MODEL_FILE, SCALER_FILE)
        
       
        input_df = get_user_input(QUESTIONS_CONFIG)
        
       
        print("\nInput Data:")
        print(input_df.to_markdown(index=False))
        
      
        has_depression = predict_depression(model, scaler, input_df)
        
       
        display_results(has_depression)
        
    except Exception as e:
        print(f"\nA fatal error occurred during execution. Please fix the issue: {e}")

🤖 Loading model and scaler...
✅ Load successful.

--- Please answer the following questions ---


Do you have little or no interest in doing things? (rate from 0-3):  0
Do you have trouble falling asleep or sleeping too much? (rate from 0-3):  0
Are you having a poor appetite or overeating problem? (rate from 0-3):  0
Are you feeling nervous or on edge? (rate from 0-3):  0
Did you find yourself to be overthinking nowadays? (rate from 0-3):  0
Did you find yourself unable to cope with all the things you had to do? (rate from 0-3):  0
Are you feeling that too much work is piling up, making you feel overwhelmed? (rate from 0-3):  0
How much would you rate your stress level? (0-40):  0
How much would you rate your anxiety level? (0-27):  0



Input Data:
|   no interest |   sleep |   appetite |   on edge |   overthinking |   cannot cope |   too much work |   stress |   anxiety |
|--------------:|--------:|-----------:|----------:|---------------:|--------------:|----------------:|---------:|----------:|
|             0 |       0 |          0 |         0 |              0 |             0 |               0 |        0 |         0 |

--- Processing Data ---
📈 Data scaled successfully.

      *** DEPRESSION PREDICTION ***
✅ The person **does not show signs of depression** (Prediction: 0).
    Remember, this is not a substitute for professional medical advice.
